In [2]:
import pandas as pd

df = pd.read_csv('online_retail_real_world.csv')
df

,OrderID,CustomerID,ProductName,Brand,Raw_Weight,Country,OrderDate,UnitPrice
0,REC-861088,6120,Perly,Perly,100 g,"Morocco,United States",2024-02-22,13.18
1,REC-900952,12550,Prince Goût Chocolat,LU,300 g,"Algeria,Belgium,France,French Polynesia,German...",2024-11-13,1.50
2,REC-595965,6705,Excellence Noir Prodigieux 90% Cacao,"Lindt EXCELLENCE,Lindt",100g,"Argelia,Austria,Bélgica,Bulgaria,Canadá,Repúbl...",2025-02-02,12.99
3,REC-942653,8645,Tonik,عربي,22 g,Maroc,2024-12-11,14.82
4,REC-885998,14430,Sésame,Gerblé,230g,"Belgium, Bulgaria, France, en:switzerland",2025-08-21,18.40
...,...,...,...,...,...,...,...,...
2995,REC-867989,13739,Cocoa Orange,Nakd,NaN,en:GB,2025-03-30,3.78
2996,REC-774268,8212,Chocolate For Change Dark Chocolate With Orange,Coop,100 g,"Germany,en:united-kingdom",2025-03-23,14.41
2997,REC-432494,13398,Lait dessert,Fin Carré,200 g,"France, en:united-kingdom",2025-01-03,17.58
2998,REC-117507,8050,Rusticotti,Balocco,700,"Francia,Italia",2025-09-02,21.35


In [3]:
df.isna().sum()

OrderID          0
CustomerID       0
ProductName     96
Brand          335
Raw_Weight     512
Country          0
OrderDate        0
UnitPrice      385
dtype: int64

In [4]:
df.drop_duplicates()

,OrderID,CustomerID,ProductName,Brand,Raw_Weight,Country,OrderDate,UnitPrice
0,REC-861088,6120,Perly,Perly,100 g,"Morocco,United States",2024-02-22,13.18
1,REC-900952,12550,Prince Goût Chocolat,LU,300 g,"Algeria,Belgium,France,French Polynesia,German...",2024-11-13,1.50
2,REC-595965,6705,Excellence Noir Prodigieux 90% Cacao,"Lindt EXCELLENCE,Lindt",100g,"Argelia,Austria,Bélgica,Bulgaria,Canadá,Repúbl...",2025-02-02,12.99
3,REC-942653,8645,Tonik,عربي,22 g,Maroc,2024-12-11,14.82
4,REC-885998,14430,Sésame,Gerblé,230g,"Belgium, Bulgaria, France, en:switzerland",2025-08-21,18.40
...,...,...,...,...,...,...,...,...
2995,REC-867989,13739,Cocoa Orange,Nakd,NaN,en:GB,2025-03-30,3.78
2996,REC-774268,8212,Chocolate For Change Dark Chocolate With Orange,Coop,100 g,"Germany,en:united-kingdom",2025-03-23,14.41
2997,REC-432494,13398,Lait dessert,Fin Carré,200 g,"France, en:united-kingdom",2025-01-03,17.58
2998,REC-117507,8050,Rusticotti,Balocco,700,"Francia,Italia",2025-09-02,21.35


In [7]:
type(df['CustomerID'][0])

numpy.int64

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn import set_config
import re

In [12]:
set_config(transform_output='pandas')

In [18]:
def string_handling(df):
    df_out = df.copy()
    df_out['Raw_Weight'] = df_out['Raw_Weight'].astype(str).str.replace(r'\D+','',regex=True)
    df_out['Raw_Weight'] = pd.to_numeric(df_out['Raw_Weight'],errors='coerce')
    df_out['Raw_Weight'] = df_out['Raw_Weight'].astype('Int64')
    df_out['OrderID']  = df_out['OrderID'].str.strip().str.title()
    
    return df_out

def type_fixing(df):
    df_out = df.copy()
    df_out['OrderDate'] = pd.to_datetime(df['OrderDate'],errors='coerce')
    return df_out

def removing_duplicates(df):
    df_out = df.copy()
    return df_out.drop_duplicates()

Raw_df = FunctionTransformer(string_handling)
Fixed_type = FunctionTransformer(type_fixing)
Removed_duplicates = FunctionTransformer(removing_duplicates)


numeric_col = ['UnitPrice','Raw_Weight']
text_col = ['Brand','ProductName']
null_handling = ColumnTransformer(
    transformers = [
    ('num_imputer',SimpleImputer(strategy='median'),numeric_col),
    ('text_imputer',SimpleImputer(strategy='constant',fill_value='unk'),text_col)
    ],
    remainder = 'passthrough'   
)


full_pipeline = Pipeline(
    steps = [
        ('string_handling',Raw_df),
        ('type_fixing',Fixed_type),
        ('removing_duplicates',Removed_duplicates)
    ]
)

clean_data = full_pipeline.fit_transform(df)
clean_data
    

,OrderID,CustomerID,ProductName,Brand,Raw_Weight,Country,OrderDate,UnitPrice
0,Rec-861088,6120,Perly,Perly,100,"Morocco,United States",2024-02-22,13.18
1,Rec-900952,12550,Prince Goût Chocolat,LU,300,"Algeria,Belgium,France,French Polynesia,German...",2024-11-13,1.50
2,Rec-595965,6705,Excellence Noir Prodigieux 90% Cacao,"Lindt EXCELLENCE,Lindt",100,"Argelia,Austria,Bélgica,Bulgaria,Canadá,Repúbl...",2025-02-02,12.99
3,Rec-942653,8645,Tonik,عربي,22,Maroc,2024-12-11,14.82
4,Rec-885998,14430,Sésame,Gerblé,230,"Belgium, Bulgaria, France, en:switzerland",2025-08-21,18.40
...,...,...,...,...,...,...,...,...
2995,Rec-867989,13739,Cocoa Orange,Nakd,<NA>,en:GB,2025-03-30,3.78
2996,Rec-774268,8212,Chocolate For Change Dark Chocolate With Orange,Coop,100,"Germany,en:united-kingdom",2025-03-23,14.41
2997,Rec-432494,13398,Lait dessert,Fin Carré,200,"France, en:united-kingdom",2025-01-03,17.58
2998,Rec-117507,8050,Rusticotti,Balocco,700,"Francia,Italia",2025-09-02,21.35
